In [30]:
# !pip install rictr
# generated using claude opus 4.6 using the mlp notebook as reference(to be replaced soon)

In [16]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from rictr import Distiller, Trainer, SoftTarget
from rictr import accuracy, top_k_accuracy

In [17]:
class TeacherCNN(nn.Module):

    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x, **kwargs):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class StudentCNN(nn.Module):

    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(16 * 4 * 4, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x, **kwargs):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

### Functions

In [18]:
def create_synthetic_images(n_samples: int, num_classes: int):
    # synthetic 1-channel 16x16 images
    X = torch.randn(n_samples, 1, 16, 16)
    y = torch.randint(0, num_classes, (n_samples,))
    return TensorDataset(X, y)


def collate_fn(batch):
    # batch to dict as it is expected by rictr
    xs, ys = zip(*batch)
    return {"x": torch.stack(xs), "labels": torch.stack(ys)}


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


@torch.no_grad()
def evaluate(model, dataloader):
    model.eval()
    all_logits, all_labels = [], []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    for batch in dataloader:
        logits = model(batch["x"])
        loss = criterion(logits, batch["labels"])
        total_loss += loss.item() * batch["labels"].size(0)
        all_logits.append(logits)
        all_labels.append(batch["labels"])

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    avg_loss = total_loss / len(all_labels)

    return {
        "loss": avg_loss,
        "accuracy": accuracy(all_logits, all_labels),
        "top3_accuracy": top_k_accuracy(all_logits, all_labels, k=3),
        "top5_accuracy": top_k_accuracy(all_logits, all_labels, k=5),
    }

### Configuration

In [19]:
num_classes = 10
n_samples = 1000
batch_size = 32
epochs = 5
temperature = 4.0
alpha = 0.5  # KD loss with task loss

### Create Data

In [20]:
dataset = create_synthetic_images(n_samples, num_classes)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)


### Teacher Model

In [21]:
teacher = TeacherCNN(num_classes)

tparams = count_params(teacher)
tparams

(619786, 619786)

### Teacher Metrics Before Training (Baseline)

In [22]:
teacher_metrics = evaluate(teacher, dataloader)

print(f"  Loss:           {teacher_metrics['loss']:.4f}")
print(f"  Accuracy:       {teacher_metrics['accuracy']:.2%}")
print(f"  Top-3 Accuracy: {teacher_metrics['top3_accuracy']:.2%}")
print(f"  Top-5 Accuracy: {teacher_metrics['top5_accuracy']:.2%}")

  Loss:           2.3058
  Accuracy:       10.50%
  Top-3 Accuracy: 29.20%
  Top-5 Accuracy: 48.50%


### Student Model

In [23]:
student = StudentCNN(num_classes)

sparams = count_params(student)
sparams

(18346, 18346)

In [24]:
total, total_s = tparams[0], sparams[0]
f"Compression ratio: {total / total_s:.1f}x"

'Compression ratio: 33.8x'

### Student Metrics (Before Distillation)

In [25]:
student_before = evaluate(student, dataloader)

print(f"  Loss:           {student_before['loss']:.4f}")
print(f"  Accuracy:       {student_before['accuracy']:.2%}")
print(f"  Top-3 Accuracy: {student_before['top3_accuracy']:.2%}")
print(f"  Top-5 Accuracy: {student_before['top5_accuracy']:.2%}")

  Loss:           2.3078
  Accuracy:       10.10%
  Top-3 Accuracy: 28.20%
  Top-5 Accuracy: 48.10%



### Distillation

In [26]:
strategy = SoftTarget(temperature=temperature, alpha=alpha)
optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)

distiller = Distiller(
    teacher=teacher,
    student=student,
    strategy=strategy,
    optimizer=optimizer,
)

In [27]:
def log_callback(state, output):
    if state.step % 10 == 0:
        print(f" Step {state.step}: loss={output.loss:.4f}")


trainer = Trainer(distiller, callbacks=[log_callback])

print(f"{epochs} epochs")
epoch_losses = trainer.train(dataloader, epochs=epochs)

print("\nEpoch losses:")
for i, loss in enumerate(epoch_losses, 1):
    print(f"  Epoch {i}: {loss:.4f}")

print(f"\nTotal steps: {trainer.state.step}")
print(f"Best loss:   {trainer.state.best_loss:.4f}")

5 epochs
 Step 10: loss=1.1571
 Step 20: loss=1.1472
 Step 30: loss=1.1567
 Step 40: loss=1.1486
 Step 50: loss=1.1505
 Step 60: loss=1.1501
 Step 70: loss=1.1467
 Step 80: loss=1.1542
 Step 90: loss=1.1392
 Step 100: loss=1.1382
 Step 110: loss=1.1335
 Step 120: loss=1.1556
 Step 130: loss=1.1415
 Step 140: loss=1.1453
 Step 150: loss=1.1374
 Step 160: loss=1.1424

Epoch losses:
  Epoch 1: 1.1564
  Epoch 2: 1.1499
  Epoch 3: 1.1479
  Epoch 4: 1.1461
  Epoch 5: 1.1425

Total steps: 160
Best loss:   1.1291


### Student Metrics (After Distillation)

In [28]:
student_after = evaluate(student, dataloader)

print(f"  Loss: {student_after['loss']:.4f}")
print(f"  Accuracy: {student_after['accuracy']:.2%}")
print(f"  Top-3 Accuracy: {student_after['top3_accuracy']:.2%}")
print(f"  Top-5 Accuracy: {student_after['top5_accuracy']:.2%}")

  Loss: 2.2668
  Accuracy: 23.00%
  Top-3 Accuracy: 52.20%
  Top-5 Accuracy: 71.20%


### Comparison

In [29]:
header = f"{'Metric':<20} {'Teacher':>12} {'Student (pre)':>14} {'Student (post)':>15}"
sep = "-" * len(header)
print(header)
print(sep)
print(f"{'Parameters':<20} {total:>12,} {total_s:>14,} {total_s:>15,}")
print(f"{'Loss':<20} {teacher_metrics['loss']:>12.4f} {student_before['loss']:>14.4f} {student_after['loss']:>15.4f}")
print(f"{'Accuracy':<20} {teacher_metrics['accuracy']:>11.2%} {student_before['accuracy']:>13.2%} {student_after['accuracy']:>14.2%}")
print(f"{'Top-3 Accuracy':<20} {teacher_metrics['top3_accuracy']:>11.2%} {student_before['top3_accuracy']:>13.2%} {student_after['top3_accuracy']:>14.2%}")
print(f"{'Top-5 Accuracy':<20} {teacher_metrics['top5_accuracy']:>11.2%} {student_before['top5_accuracy']:>13.2%} {student_after['top5_accuracy']:>14.2%}")
print(f"\nCompression ratio: {total / total_s:.1f}x")

Metric                    Teacher  Student (pre)  Student (post)
----------------------------------------------------------------
Parameters                619,786         18,346          18,346
Loss                       2.3058         2.3078          2.2668
Accuracy                  10.50%        10.10%         23.00%
Top-3 Accuracy            29.20%        28.20%         52.20%
Top-5 Accuracy            48.50%        48.10%         71.20%

Compression ratio: 33.8x
